In [102]:
import polars as pl
import numpy as np
from src.storage.ch_client import ch_manager
from sklearn.linear_model import LinearRegression
import os

In [103]:
ch_client = ch_manager.market_db

# sql = """
#     SELECT 
#         t.timestamp,
#         t.symbol,
#         t.price as trade_price,
#         t.amount,
#         (ob.bid_prices[1] * ob.ask_volumes[1] + ob.ask_prices[1] * ob.bid_volumes[1]) / nullIf(ob.bid_volumes[1] + ob.ask_volumes[1],0) as prev_micro_price,
#         (ob.bid_volumes[1] - ob.ask_volumes[1]) / nullIf(ob.bid_volumes[1] + ob.ask_volumes[1],0) as imbalance,
#         ob.bid_prices[1] as best_bid,
#         ob.ask_prices[1] as best_ask,
#         t.timestamp - ob.timestamp AS data_lag_ms
#     FROM (
#         SELECT * FROM market_data.trades 
#         WHERE exchange_id = 'binance' AND symbol = 'BTC/USDT' 
#         AND toDate(fromUnixTimestamp64Milli(timestamp)) = '2026-03-21'
#     ) AS t
#     ASOF LEFT JOIN (
#         SELECT * FROM market_data.orderbook 
#         WHERE exchange_id = 'binance' AND symbol = 'BTC/USDT' 
#         AND toDate(fromUnixTimestamp64Milli(timestamp)) = '2026-03-21'
#     ) AS ob
#         ON t.exchange_id=ob.exchange_id
#         AND t.symbol=ob.symbol
#         AND t.timestamp >= ob.timestamp
# """

# arrow_table = ch_client.query_arrow(sql)

# df = pl.from_arrow(arrow_table)

# del arrow_table

# clear_df = df.filter(pl.col('data_lag_ms').abs() <= 100)

# impact_df = clear_df.with_columns([
#     ((pl.col('best_ask') - pl.col('trade_price')) / pl.col('best_ask') * 1000).alias('slippage_bps_buy'),
#     ((pl.col('trade_price') - pl.col('best_bid')) / pl.col('best_bid') * 1000).alias('slippage_bps_sell'),
#     (pl.col('trade_price') * pl.col('amount')).alias('notional_usdt')
# ])

# size_buckets = impact_df.with_columns([
#     pl.when(pl.col('notional_usdt') < 1000).then(pl.lit('Retail (<1k)')).when(pl.col('notional_usdt') < 10000).then(pl.lit('Pro (1k-10k)')).otherwise(pl.lit('Whale (>10k)')).alias('trader_category')
# ]).group_by('trader_category').agg([
#     pl.col('slippage_bps_buy').mean().alias('avg_impact_cost_buy'),
#     pl.col('slippage_bps_sell').mean().alias('avg_impact_cost_sell'),
#     pl.len().alias('trade_count')
# ])

# print(size_buckets)


In [ ]:
ob_sql = """
    SELECT
        timestamp,
        symbol,
        bid_prices,
        bid_volumes,
        ask_prices,
        ask_volumes,
        (bid_prices[1] + ask_prices[1]) / 2 as mid_price
    FROM market_data.orderbook
    WHERE exchange_id='binance' 
    AND symbol='BTC/USDT'
    AND toDate(fromUnixTimestamp64Milli(timestamp)) = '2026-03-22'
    ORDER BY timestamp ASC
    LIMIT 100000
"""

ob_arrow = ch_client.query_arrow(ob_sql)
ob_df = pl.from_arrow(ob_arrow)

timestamp,symbol,bid_prices,bid_volumes,ask_prices,ask_volumes,mid_price
i64,str,list[f64],list[f64],list[f64],list[f64],f64
1774137600014,"""BTC/USDT""","[68918.11, 68918.1, … 68911.75]","[2.44117, 0.00016, … 0.31331]","[68918.12, 68918.13, … 68925.64]","[0.11993, 0.00056, … 0.00008]",68918.115
1774137600114,"""BTC/USDT""","[68918.12, 68918.11, … 68911.69]","[2.99221, 1.70549, … 0.0003]","[68918.13, 68918.14, … 68925.64]","[0.00127, 0.00056, … 0.00008]",68918.125
1774137600214,"""BTC/USDT""","[68924.08, 68924.07, … 68914.42]","[1.66775, 0.00016, … 0.003]","[68924.09, 68924.1, … 68928.0]","[0.9627, 0.0004, … 0.0012]",68924.085
1774137600314,"""BTC/USDT""","[68924.08, 68924.07, … 68914.42]","[1.54071, 0.00016, … 0.003]","[68924.09, 68924.1, … 68928.0]","[0.82518, 0.00048, … 0.0012]",68924.085
1774137600414,"""BTC/USDT""","[68924.08, 68924.07, … 68915.07]","[1.54477, 0.00016, … 1.39326]","[68924.09, 68924.1, … 68928.02]","[0.74427, 0.00064, … 0.00008]",68924.085
1774137600515,"""BTC/USDT""","[68918.2, 68918.18, … 68915.0]","[0.00016, 0.00016, … 0.00016]","[68918.21, 68922.38, … 68927.36]","[0.92185, 0.0936, … 0.0001]",68918.205
1774137600614,"""BTC/USDT""","[68905.95, 68905.94, … 68900.48]","[0.2952, 0.00016, … 0.00008]","[68905.96, 68905.97, … 68914.6]","[3.17912, 0.00016, … 0.00008]",68905.955
1774137600714,"""BTC/USDT""","[68905.95, 68905.94, … 68900.46]","[0.42099, 0.00032, … 0.00016]","[68905.96, 68905.97, … 68914.6]","[3.20968, 0.00016, … 0.00008]",68905.955
1774137600814,"""BTC/USDT""","[68905.95, 68905.94, … 68900.19]","[0.28562, 0.00032, … 0.00145]","[68905.96, 68905.97, … 68912.91]","[3.20968, 0.00016, … 0.10034]",68905.955


In [105]:
def calculate_vamp(df:pl.DataFrame,depth=5):
    return df.with_columns([
        (
            (pl.col('bid_prices').list.slice(0,depth) * pl.col('bid_volumes').list.slice(0,depth)).list.sum() / pl.col('bid_volumes').list.slice(0,depth).list.sum()
        ).alias(f'vamp_bid_{depth}'),
        (
            (pl.col('ask_prices').list.slice(0,depth) * pl.col('ask_volumes').list.slice(0,depth)).list.sum() / pl.col('ask_volumes').list.slice(0,depth).list.sum()
        ).alias(f'vamp_ask_{depth}')
    ]).with_columns([
        ((pl.col(f'vamp_bid_{depth}') + pl.col(f'vamp_ask_{depth}')) / 2).alias('vamp')
    ])

In [106]:
df = calculate_vamp(ob_df)
df = df.with_columns([
    ((pl.col('vamp') - pl.col('mid_price')) / pl.col('mid_price') * 10000).alias('vamp_bias_bp'),
    ((pl.col('mid_price').shift(-5) - pl.col('mid_price')) / pl.col('mid_price') * 10000).alias('target_5_tick'),
    ((pl.col('mid_price').shift(-10) - pl.col('mid_price')) / pl.col('mid_price') * 10000).alias('target_10_tick'),
    ((pl.col('mid_price').shift(-20) - pl.col('mid_price')) / pl.col('mid_price') * 10000).alias('target_20_tick'),
    ((pl.col('mid_price').shift(-50) - pl.col('mid_price')) / pl.col('mid_price') * 10000).alias('target_50_tick'),
    ((pl.col('mid_price').shift(-100) - pl.col('mid_price')) / pl.col('mid_price') * 10000).alias('target_100_tick')
])
target_ticks = {}
for lag in [5,10,20,50,100]:
    correlation = df.drop_nulls().select(pl.corr('vamp_bias_bp',f'target_{lag}_tick')).item()
    correlation = abs(correlation)
    target_ticks[f'target_{lag}_tick'] = correlation
    print(f"{lag}因子IC值: {correlation:.4f}")

max_key = max(target_ticks,key=target_ticks.get)


5因子IC值: 0.0150
10因子IC值: 0.0049
20因子IC值: 0.0188
50因子IC值: 0.0162
100因子IC值: 0.0080


In [107]:
def calculate_ofi_v1(df:pl.DataFrame):
    return df.with_columns([
        pl.col('bid_prices').list.get(0).alias('p_b'),
        pl.col('bid_volumes').list.get(0).alias('v_b'),
        pl.col('ask_prices').list.get(0).alias('p_a'),
        pl.col('ask_volumes').list.get(0).alias('v_a')
    ]).with_columns([
        pl.when(pl.col('p_b') > pl.col('p_b').shift(1)).then(pl.col('v_b')).when(pl.col('p_b') == pl.col('p_b').shift(1)).then(pl.col('v_b') - pl.col('v_b').shift(1)).otherwise(-pl.col('v_b').shift(1)).alias('db'),
        pl.when(pl.col('p_a') < pl.col('p_a').shift(1)).then(pl.col('v_a')).when(pl.col('p_a') == pl.col('p_b').shift(1)).then(pl.col('v_a') - pl.col('v_a').shift(1)).otherwise(-pl.col('v_a').shift(1)).alias('da')
    ]).with_columns([
        (pl.col('db') - pl.col('da')).alias('ofi_raw')
    ]).with_columns([
        pl.col('ofi_raw').rolling_mean(window_size=20).alias('factor_ofi_smooth')
    ])

df_ofi = calculate_ofi_v1(df)
ic_ofi = df_ofi.drop_nulls().select(pl.corr('factor_ofi_smooth',max_key)).item()
print(f"{max_key} OFI 因子 IC 值: {ic_ofi:.4f}")

target_20_tick OFI 因子 IC 值: -0.0820


In [108]:
train_df = df_ofi.select(['vamp_bias_bp','factor_ofi_smooth','target_20_tick']).drop_nulls()

X = train_df.select(['vamp_bias_bp','factor_ofi_smooth']).to_numpy()
y = train_df.select(['target_20_tick']).to_numpy()

model = LinearRegression()
model.fit(X,y)

w1,w2 = model.coef_[0]
intercept = model.intercept_[0]
print(model.coef_)

print(f"最优权重 w1 (VAMP): {w1:.6f}")
print(f"最优权重 w2 (ofi): {w2:.6f}")
print(f"截距 b: {intercept:.6f}")

[[-1.45182852 -0.10724449]]
最优权重 w1 (VAMP): -1.451829
最优权重 w2 (ofi): -0.107244
截距 b: 0.143628


In [109]:
df_ofi = df_ofi.with_columns([
    (pl.col('vamp_bias_bp') * w1 + pl.col('factor_ofi_smooth') * w2 + intercept).alias('combined_alpha')
])

combined_ic = df_ofi.drop_nulls().select(pl.corr('combined_alpha','target_20_tick')).item()
print(f"复合信号的 IC 值: {combined_ic:.4f}")

复合信号的 IC 值: 0.0845
